# Notebook 06: Grad-CAM Explainability

**Student:** Okidi Patrovas Gabriel | 2025/HD07/26018U  
**Institution:** Makerere University, Kampala, Uganda  
**Course:** MSB7216 — Deep Learning for Health Data

## Overview
This notebook applies Gradient-weighted Class Activation Mapping (Grad-CAM) to the best-performing model from Notebooks 04 and 05. Grad-CAM produces a heatmap showing which regions of an input image most influenced the model's prediction. We analyse these heatmaps per class and assess whether highlighted regions correspond to known histological markers.

## Objectives
1. Load the best model weights automatically from Drive.
2. Load test images from local Colab storage.
3. Generate Grad-CAM heatmaps for correctly classified images from all five classes.
4. Generate heatmaps for misclassified images to understand failure modes.
5. Produce a detailed per-class three-column figure for the report.
6. Save all figures to Drive.

In [ ]:
# ── Cell 1: Install and Import ────────────────────────────────────────────────
!pip install torch torchvision grad-cam --quiet

import os, random, warnings, json, shutil
import concurrent.futures
warnings.filterwarnings('ignore')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image
import time

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

from google.colab import drive

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: GPU not available. Go to Runtime > Change runtime type > GPU')

In [ ]:
# ── Cell 2: Mount Drive, Load Splits and Identify Best Model ──────────────────
drive.mount('/content/drive', force_remount=False)

BASE_DIR    = Path('/content/drive/MyDrive/lung-colon-cancer-histopathology')
FIGURES_DIR = BASE_DIR / 'figures'
MODELS_DIR  = BASE_DIR / 'models'

with open(BASE_DIR / 'data' / 'dataset_splits.json', 'r') as f:
    split_data = json.load(f)

train_paths  = split_data['train_paths']
val_paths    = split_data['val_paths']
test_paths   = split_data['test_paths']
train_labels = split_data['train_labels']
val_labels   = split_data['val_labels']
test_labels  = split_data['test_labels']
CLASS_NAMES  = split_data['class_names']

CLASS_FULL_NAMES = {
    'colon_aca': 'Colon Adenocarcinoma',
    'colon_n'  : 'Benign Colon Tissue',
    'lung_aca' : 'Lung Adenocarcinoma',
    'lung_n'   : 'Benign Lung Tissue',
    'lung_scc' : 'Lung Squamous Cell Carcinoma',
}

# Automatically identify best model from saved results
results_files = {
    'EfficientNet-B0': MODELS_DIR / 'efficientnet_b0_results.json',
    'ResNet-50'      : MODELS_DIR / 'resnet50_results.json',
}

best_model_name = None
best_auc        = 0.0

for name, path in results_files.items():
    if path.exists():
        with open(path) as f:
            r = json.load(f)
        if r['macro_auc'] > best_auc:
            best_auc        = r['macro_auc']
            best_model_name = name

print(f'Best model : {best_model_name} (Macro AUC: {best_auc:.4f})')
print(f'Test images: {len(test_paths)}')

In [ ]:
# ── Cell 3: Copy Test Images to Local Storage ─────────────────────────────────
# Grad-CAM loads images one by one. Local storage prevents Drive timeout.

LOCAL_DATA = Path('/content/local_data')

existing     = len(list(LOCAL_DATA.rglob('*.jpeg'))) if LOCAL_DATA.exists() else 0
total_needed = len(train_paths) + len(val_paths) + len(test_paths)

if existing >= total_needed:
    print(f'Local dataset already complete: {existing} files. Skipping copy.')
else:
    print(f'Found {existing}/{total_needed} files. Copying missing files...')

    all_paths_combined  = train_paths + val_paths + test_paths
    all_labels_combined = train_labels + val_labels + test_labels

    copy_tasks = []
    for src_path, label in zip(all_paths_combined, all_labels_combined):
        class_name = CLASS_NAMES[label]
        dst_dir    = LOCAL_DATA / class_name
        dst_dir.mkdir(parents=True, exist_ok=True)
        dst_path   = dst_dir / Path(src_path).name
        if not dst_path.exists():
            copy_tasks.append((str(src_path), str(dst_path)))

    if copy_tasks:
        start = time.time()
        def copy_file(args):
            src, dst = args
            shutil.copy2(src, dst)
            return dst
        with concurrent.futures.ThreadPoolExecutor(max_workers=16) as executor:
            list(executor.map(copy_file, copy_tasks))
        elapsed = time.time() - start
        print(f'Done. {len(list(LOCAL_DATA.rglob("*.jpeg")))} files in {elapsed:.0f}s.')

# Update test paths to local storage
test_paths_local = [
    str(LOCAL_DATA / CLASS_NAMES[l] / Path(p).name)
    for p, l in zip(test_paths, test_labels)
]

print(f'Test paths updated : {len(test_paths_local)}')
print(f'Reading from       : {test_paths_local[0]}')

In [ ]:
# ── Cell 4: Load Best Model ───────────────────────────────────────────────────

if best_model_name == 'EfficientNet-B0':
    model = models.efficientnet_b0(weights=None)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 5)
    )
    checkpoint_path = MODELS_DIR / 'efficientnet_b0_best.pth'
    target_layers   = [model.features[-1]]

else:  # ResNet-50
    model = models.resnet50(weights=None)
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, 5)
    )
    checkpoint_path = MODELS_DIR / 'resnet50_best.pth'
    target_layers   = [model.layer4[-1]]

model.load_state_dict(torch.load(checkpoint_path, map_location=DEVICE))
model = model.to(DEVICE)
model.eval()

print(f'{best_model_name} weights loaded.')
print(f'Checkpoint : {checkpoint_path}')
print(f'Target layer : {target_layers}')

In [ ]:
# ── Cell 5: Define Transforms and Grad-CAM Helper ────────────────────────────

IMAGE_SIZE    = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def generate_gradcam(img_path, target_class, model, target_layers, device=DEVICE):
    """
    Generate a Grad-CAM heatmap for a given image and target class.
    Returns (original_rgb, cam_overlay) both as numpy arrays.
    """
    img_pil      = Image.open(img_path).convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    original_rgb = np.array(img_pil).astype(np.float32) / 255.0
    input_tensor = eval_transform(img_pil).unsqueeze(0).to(device)

    cam          = GradCAM(model=model, target_layers=target_layers)
    targets      = [ClassifierOutputTarget(target_class)]
    grayscale    = cam(input_tensor=input_tensor, targets=targets)[0]
    overlay      = show_cam_on_image(original_rgb, grayscale, use_rgb=True)

    return original_rgb, overlay, grayscale


print('Grad-CAM helper function defined.')

In [ ]:
# ── Cell 6: Collect Correct and Incorrect Samples ────────────────────────────
# Collect 3 correctly classified and up to 2 misclassified images per class

model.eval()
correct_samples   = {c: [] for c in range(5)}
incorrect_samples = {c: [] for c in range(5)}

print('Scanning test images...')

with torch.no_grad():
    for i, (img_path, label) in enumerate(zip(test_paths_local, test_labels)):
        img_pil = Image.open(img_path).convert('RGB')
        tensor  = eval_transform(img_pil).unsqueeze(0).to(DEVICE)
        output  = model(tensor)
        pred    = output.argmax(1).item()

        if pred == label and len(correct_samples[label]) < 3:
            correct_samples[label].append((img_path, label, pred))
        elif pred != label and len(incorrect_samples[label]) < 2:
            incorrect_samples[label].append((img_path, label, pred))

        # Stop once we have enough samples
        if (all(len(v) >= 3 for v in correct_samples.values()) and
            all(len(v) >= 1 for v in incorrect_samples.values())):
            break

print('Sample collection complete.')
print()
for c in range(5):
    print(f'{CLASS_NAMES[c]:<15} Correct: {len(correct_samples[c])}  '
          f'Incorrect: {len(incorrect_samples[c])}')

In [ ]:
# ── Cell 7: Figure 1 — Per-Class Grad-CAM Grid (Correct Classifications) ──────

fig, axes = plt.subplots(5, 6, figsize=(22, 18))
fig.suptitle(
    f'{best_model_name} — Grad-CAM: Correctly Classified Images (3 per Class)',
    fontsize=14, y=1.01
)

col_titles = ['Original', 'Grad-CAM'] * 3
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=9)

for row, class_idx in enumerate(range(5)):
    class_name = CLASS_NAMES[class_idx]
    samples    = correct_samples[class_idx]
    col        = 0

    for sample in samples[:3]:
        img_path, true_label, pred = sample
        original, cam_img, _ = generate_gradcam(
            img_path, class_idx, model, target_layers
        )
        axes[row, col].imshow(original)
        axes[row, col].axis('off')
        if col == 0:
            axes[row, col].set_ylabel(
                CLASS_FULL_NAMES[class_name], fontsize=8, rotation=90, labelpad=4
            )
        axes[row, col + 1].imshow(cam_img)
        axes[row, col + 1].axis('off')
        col += 2

plt.tight_layout()
save_path = FIGURES_DIR / '06_gradcam_correct_grid.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

In [ ]:
# ── Cell 8: Figure 2 — Grad-CAM on Misclassified Images ──────────────────────

available_incorrect = {c: v for c, v in incorrect_samples.items() if len(v) > 0}
n_rows = len(available_incorrect)

if n_rows > 0:
    fig, axes = plt.subplots(n_rows, 4, figsize=(16, 4 * n_rows))
    if n_rows == 1:
        axes = axes[np.newaxis, :]
    fig.suptitle(f'{best_model_name} — Grad-CAM: Misclassified Images', fontsize=13)

    for row, (class_idx, samples) in enumerate(available_incorrect.items()):
        img_path, true_label, pred = samples[0]
        true_name = CLASS_NAMES[true_label]
        pred_name = CLASS_NAMES[pred]

        original, cam_true, _ = generate_gradcam(
            img_path, true_label, model, target_layers
        )
        _, cam_pred, _ = generate_gradcam(
            img_path, pred, model, target_layers
        )

        axes[row, 0].imshow(original)
        axes[row, 0].set_title(f'Original\nTrue: {true_name}', fontsize=8)
        axes[row, 0].axis('off')

        axes[row, 1].imshow(cam_true)
        axes[row, 1].set_title(f'Grad-CAM (True: {true_name})', fontsize=8)
        axes[row, 1].axis('off')

        axes[row, 2].imshow(original)
        axes[row, 2].set_title(f'Original\nPredicted: {pred_name}', fontsize=8)
        axes[row, 2].axis('off')

        axes[row, 3].imshow(cam_pred)
        axes[row, 3].set_title(f'Grad-CAM (Predicted: {pred_name})', fontsize=8)
        axes[row, 3].axis('off')

    plt.tight_layout()
    save_path = FIGURES_DIR / '06_gradcam_misclassified.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Figure saved to {save_path}')
else:
    print('No misclassified samples found in the reviewed test images.')

In [ ]:
# ── Cell 9: Figure 3 — Detailed Three-Column Grad-CAM per Class ───────────────
# Original | Heatmap | Overlay — one representative image per class

fig, axes = plt.subplots(5, 3, figsize=(12, 20))
fig.suptitle(
    f'{best_model_name} — Grad-CAM Detail: One Image per Class',
    fontsize=13
)

col_titles = ['Original Image', 'Grad-CAM Heatmap', 'Overlay']
for col, title in enumerate(col_titles):
    axes[0, col].set_title(title, fontsize=10)

for row, class_idx in enumerate(range(5)):
    class_name = CLASS_NAMES[class_idx]
    samples    = correct_samples[class_idx]
    if not samples:
        continue

    img_path                    = samples[0][0]
    original, overlay, grayscale = generate_gradcam(
        img_path, class_idx, model, target_layers
    )
    heatmap = plt.get_cmap('jet')(grayscale)[:, :, :3]

    axes[row, 0].imshow(original)
    axes[row, 0].set_ylabel(
        CLASS_FULL_NAMES[class_name], fontsize=9, rotation=90, labelpad=4
    )
    axes[row, 0].axis('off')

    axes[row, 1].imshow(heatmap)
    axes[row, 1].axis('off')

    axes[row, 2].imshow(overlay)
    axes[row, 2].axis('off')

plt.tight_layout()
save_path = FIGURES_DIR / '06_gradcam_detail_per_class.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Figure saved to {save_path}')

In [ ]:
# ── Cell 10: Summary of All Figures Saved ────────────────────────────────────

figures = sorted(FIGURES_DIR.glob('*.png'))
print(f'Total figures saved to Drive : {len(figures)}')
print()
for fig_path in figures:
    size_kb = os.path.getsize(fig_path) // 1024
    print(f'  {fig_path.name:<50} {size_kb} KB')

In [ ]:
# ── Cell 11: Save Notebook to Drive ──────────────────────────────────────────

try:
    shutil.copy('/content/06_gradcam_explainability.ipynb',
                str(BASE_DIR / 'notebooks' / '06_gradcam_explainability.ipynb'))
    print('Notebook saved to Drive.')
except:
    print('Use File > Save a copy in Drive to save this notebook.')